In [40]:
import pandas as pd

campaigns = pd.read_csv('data/campaigns.csv')
customers = pd.read_csv('data/customers.csv')
events = pd.read_csv('data/events.csv')
products = pd.read_csv('data/products.csv')
transactions = pd.read_csv('data/transactions.csv')

In [38]:
campaigns.shape, customers.shape, events.shape, products.shape, transactions.shape

((50, 7), (100000, 7), (2000000, 12), (2000, 6), (103127, 9))

In [39]:
transactions.info()
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 103127 entries, 0 to 103126
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    103127 non-null  int64  
 1   timestamp         103127 non-null  str    
 2   customer_id       103127 non-null  int64  
 3   product_id        92678 non-null   float64
 4   quantity          103127 non-null  int64  
 5   discount_applied  103127 non-null  float64
 6   gross_revenue     92678 non-null   float64
 7   campaign_id       103127 non-null  int64  
 8   refund_flag       103127 non-null  int64  
dtypes: float64(3), int64(5), str(1)
memory usage: 7.1 MB
<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column                Dtype  
---  ------                -----  
 0   event_id              int64  
 1   timestamp             str    
 2   customer_id           int64  
 3   session_id            int64  
 4 

In [6]:
transactions['timestamp'] = pd.to_datetime(transactions['timestamp'])
events['timestamp'] = pd.to_datetime(events['timestamp'])

In [7]:
transactions.isna().sum()

transaction_id          0
timestamp               0
customer_id             0
product_id          10449
quantity                0
discount_applied        0
gross_revenue       10449
campaign_id             0
refund_flag             0
dtype: int64

In [8]:
events.isna().sum()

event_id                     0
timestamp                    0
customer_id                  0
session_id                   0
event_type                   0
product_id              200371
device_type              40300
traffic_source               0
campaign_id                  0
page_category                0
session_duration_sec         0
experiment_group             0
dtype: int64

In [9]:
events['event_type'].value_counts()

event_type
view           1043573
click           379008
add_to_cart     284370
bounce          189922
purchase        103127
Name: count, dtype: int64

In [10]:
transactions = transactions.dropna(subset=['product_id', 'gross_revenue'])

In [11]:
transactions.isna().sum()
transactions.shape

(92678, 9)

In [12]:
events['event_type'].value_counts(normalize=True)

event_type
view           0.521787
click          0.189504
add_to_cart    0.142185
bounce         0.094961
purchase       0.051563
Name: proportion, dtype: float64

In [13]:
views = 1043573
clicks = 379008
carts = 284370
purchases = 103127

print("view → click:", clicks / views)
print("click → cart:", carts / clicks)
print("cart → purchase:", purchases / carts)
print("view → purchase:", purchases / views)

view → click: 0.3631830260077637
click → cart: 0.7503007852077
cart → purchase: 0.3626507718817034
view → purchase: 0.09882106953706161


In [14]:
views = events[events['event_type'] == 'view'].shape[0]
clicks = events[events['event_type'] == 'click'].shape[0]
carts = events[events['event_type'] == 'add_to_cart'].shape[0]
purchases = events[events['event_type'] == 'purchase'].shape[0]

print(views, clicks, carts, purchases)

print("view → click:", clicks / views)
print("click → cart:", carts / clicks)
print("cart → purchase:", purchases / carts)
print("view → purchase:", purchases / views)

1043573 379008 284370 103127
view → click: 0.3631830260077637
click → cart: 0.7503007852077
cart → purchase: 0.3626507718817034
view → purchase: 0.09882106953706161


In [16]:
events['traffic_source'] = events['traffic_source'].str.strip().str.lower()

In [17]:
# считаем события по источникам
funnel = (
    events
    .groupby(['traffic_source', 'event_type'])
    .size()
    .unstack(fill_value=0)
)

# считаем конверсию
funnel['conversion_view_to_purchase'] = funnel['purchase'] / funnel['view']

funnel.sort_values('conversion_view_to_purchase', ascending=False)

event_type,add_to_cart,bounce,click,purchase,view,conversion_view_to_purchase
traffic_source,,,,,,
email,40922,27251,54591,26636,150240,0.177290
paid search,54516,36715,73653,33633,201321,0.167062
social,41706,27774,55449,21903,153439,0.142747
direct,29368,19467,38987,4200,107740,0.038983
organic,117858,78715,156328,16755,430833,0.038890


In [18]:
merged = events.merge(transactions, on=['customer_id', 'timestamp'], how='inner')

In [19]:
merged.groupby('traffic_source')['gross_revenue'].sum().sort_values(ascending=False)

traffic_source
paid search    2731004.44
email          2188155.33
social         1764686.55
organic        1358042.14
direct          332077.90
Name: gross_revenue, dtype: float64

In [21]:
purchases_events = events[events['event_type'] == 'purchase']

merged = purchases_events.merge(
    transactions,
    on=['customer_id', 'product_id', 'timestamp'],
    how='inner'
)

merged.shape

(92678, 18)

In [23]:
print("purchases_events:", purchases_events.shape)
print("transactions:", transactions.shape)
print("merged:", merged.shape)

purchases_events: (103127, 12)
transactions: (92678, 9)
merged: (92678, 18)


In [24]:
merged.groupby('traffic_source').agg(
    purchases=('transaction_id', 'count'),
    revenue=('gross_revenue', 'sum'),
    aov=('gross_revenue', 'mean')
).sort_values('revenue', ascending=False)

,purchases,revenue,aov
traffic_source,,,
paid search,30218,2731004.44,90.376744
email,24013,2188155.33,91.123780
social,19658,1764686.55,89.769384
organic,15034,1358042.14,90.331392
direct,3755,332077.90,88.436192


In [30]:
merged = merged.rename(columns={
    'campaign_id_y': 'campaign_id'
})

In [32]:
merged.groupby('campaign_id').agg(
    purchases=('transaction_id', 'count'),
    revenue=('gross_revenue', 'sum'),
    aov=('gross_revenue', 'mean')
).sort_values('revenue', ascending=False).head(10)

,purchases,revenue,aov
campaign_id,,,
0,18789,1690120.04,89.952634
18,2027,187497.20,92.499852
5,2012,184244.86,91.572992
29,1995,180315.11,90.383514
44,2019,175026.54,86.689718
7,1929,171224.62,88.763411
25,1804,169397.56,93.901086
17,1934,169313.35,87.545683
16,1815,166515.24,91.743934


In [33]:
merged.groupby(['campaign_id', 'traffic_source']).agg(
    revenue=('gross_revenue', 'sum')
).sort_values('revenue', ascending=False).head(10)

revenue
campaign_id traffic_source            
0           organic         1358042.14
            direct           332077.90
5           paid search       74472.23
18          paid search       73794.28
44          paid search       73659.65
29          paid search       72099.74
7           paid search       70545.70
17          paid search       69725.79
25          paid search       69375.42
49          paid search       67948.63

In [34]:
merged[merged['campaign_id'] != 0] \
.groupby(['campaign_id', 'traffic_source']) \
.agg(revenue=('gross_revenue', 'sum')) \
.sort_values('revenue', ascending=False) \
.head(10)

,,revenue
campaign_id,traffic_source,
5,paid search,74472.23
18,paid search,73794.28
44,paid search,73659.65
29,paid search,72099.74
7,paid search,70545.70
17,paid search,69725.79
25,paid search,69375.42
49,paid search,67948.63
16,paid search,67360.23
